# Benchmarking Ordering Policies\n\nThis notebook compares different ordering policies using the DeepBullwhip benchmarking framework.\n\nWe compare:\n- **Order-Up-To (OUT)**: Standard base-stock policy\n- **POUT (alpha=0.3)**: Proportional OUT with aggressive smoothing\n- **POUT (alpha=0.7)**: Proportional OUT with moderate smoothing\n- **Constant Order**: Fixed quantity baseline (BWR=0 by construction)

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from deepbullwhip.benchmark import BenchmarkRunner
from deepbullwhip.diagnostics.plots import COLORS, _apply_style, DOUBLE_COL, GOLDEN

_apply_style()

runner = BenchmarkRunner(
    chain_config="semiconductor_4tier",
    demand="semiconductor_ar1",
    T=156,
    N=50,
    seed=42,
)

results = runner.run(
    policies=[
        "order_up_to",
        ("proportional_out", {"alpha": 0.3}),
        ("proportional_out", {"alpha": 0.7}),
        ("constant_order", {"order_quantity": 11.6}),
    ],
    forecasters=["naive"],
    metrics=["BWR", "FILL_RATE", "TC"],
)

print(results.to_string(index=False))

## Pivot Table: Policy x Metric

In [ ]:
pivot = results.pivot_table(
    index=["policy", "echelon"],
    columns="metric",
    values="value",
    aggfunc="mean",
)
print(pivot.to_string(float_format="%.3f"))

## Comparison vs OUT Baseline

In [ ]:
comparison = runner.compare(results, baseline="order_up_to")
comp_pivot = comparison.pivot_table(
    index=["policy", "echelon"],
    columns="metric",
    values="pct_change",
    aggfunc="mean",
)
print("Percentage change vs OUT baseline:")
print(comp_pivot.to_string(float_format="%+.1f%%"))

## BWR vs Fill Rate Tradeoff (Echelon 1)\n\nThe key tradeoff: lower alpha reduces bullwhip but may reduce fill rate.

In [ ]:
POLICY_COLORS = [COLORS["E1"], COLORS["E2"], COLORS["E3"], COLORS["E4"]]
POLICY_LABELS = {"order_up_to": "OUT", "proportional_out": "POUT", "constant_order": "Constant"}

e1 = results[results["echelon"] == "E1"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(DOUBLE_COL, DOUBLE_COL / GOLDEN / 1.8))

for metric, ax, ylabel in [("BWR", ax1, "Bullwhip Ratio"), ("FILL_RATE", ax2, "Fill Rate")]:
    data = e1[e1["metric"] == metric].copy()
    data = data.sort_values("value", ascending=True)
    labels = [POLICY_LABELS.get(p, p) for p in data["policy"]]
    colors = [POLICY_COLORS[i % len(POLICY_COLORS)] for i in range(len(data))]
    
    bars = ax.barh(labels, data["value"], color=colors, edgecolor="white", linewidth=0.5, height=0.55)
    ax.set_xlabel(ylabel)
    
    # Value labels on bars
    for bar, val in zip(bars, data["value"]):
        ax.text(bar.get_width() + ax.get_xlim()[1] * 0.02, bar.get_y() + bar.get_height() / 2,
                f"{val:.2f}", va="center", fontsize=7, color="#333333")

if ax1.get_xlim()[1] > 1:
    ax1.axvline(x=1.0, color=COLORS["grid"], linestyle="--", linewidth=0.5)

fig.savefig("benchmark_policies_tradeoff.pdf", dpi=300, bbox_inches="tight")
fig.savefig("benchmark_policies_tradeoff.png", dpi=300, bbox_inches="tight")
plt.show()

## Export LaTeX Table

In [ ]:
from deepbullwhip.benchmark.report import to_latex

latex = to_latex(results, caption="Policy Comparison on Semiconductor 4-Tier Chain", label="tab:policy-comparison")
print(latex)